In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
DATA_DIR = "/kaggle/input/datasets/yonatankkiross/daily-demand-and-supplier-config/"

In [ ]:
import pandas as pd
import os

In [ ]:
DATA_DIR = "/kaggle/input/datasets/yonatankkiross/daily-demand-and-supplier-config/"
demand = pd.read_csv(os.path.join(DATA_DIR, "02_daily_demand.csv"))
products = pd.read_csv(os.path.join(DATA_DIR, "03_product_inventory.csv"))
suppliers = pd.read_csv(os.path.join(DATA_DIR, "04_supplier_config.csv"))

In [ ]:
print("DAILY DEMAND")
display(demand.head())

In [ ]:
print("DAILY DEMAND")
display(demand.tail())

In [ ]:
print("Shape:", demand.shape)
print("Columns:", demand.columns.tolist())

In [ ]:
print("\nPRODUCT INVENTORY")
display(products)

In [ ]:
print("Columns:", products.columns.tolist())

In [ ]:
print("\nSUPPLIER CONFIGURATION")
display(suppliers)

In [ ]:
print("Columns:", suppliers.columns.tolist())

**Check missing values**

In [ ]:
print("Missing values in daily demand:")
print(demand.isnull().sum())

In [ ]:
print("\nMissing values in product inventory:")
print(products.isnull().sum())

In [ ]:
print("\nMissing values in supplier configuration:")
print(suppliers.isnull().sum())

**Check duplicate date/product records**

In [ ]:
print("\nDuplicate date-product records:",
      demand.duplicated(subset=["date", "product"]).sum())

**Check demand statistics for each product**

In [ ]:
demand_summary = (
    demand.groupby("product")["demand"]
    .agg(["count", "mean", "std", "min", "max"])
    .round(2)
)

display(demand_summary)

**Confirm dates and product coverage**

In [ ]:
demand["date"] = pd.to_datetime(demand["date"])

print("Date range:", demand["date"].min(), "to", demand["date"].max())
print("Products:", sorted(demand["product"].unique()))
print("Rows per product:")
print(demand.groupby("product").size())

**Simulate Day 1 with no new orders**

**Starting inventory from your product CSV**

In [ ]:
inventory = products.set_index("product")["initial_inventory"].to_dict()

**Get demand for the first day**

In [ ]:
first_day = demand[demand["date"] == pd.Timestamp("2026-01-01")].copy()

**Calculate what happens when we place no orders**

In [ ]:
first_day["starting_stock"] = first_day["product"].map(inventory)
first_day["units_sold"] = first_day[
    ["demand", "starting_stock"]
].min(axis=1)

first_day["missed_sales"] = (
    first_day["demand"] - first_day["units_sold"]
)

first_day["ending_stock"] = (
    first_day["starting_stock"] - first_day["units_sold"]
)

first_day["revenue"] = (
    first_day["units_sold"] * first_day["selling_price"]
)

first_day["holding_cost_total"] = (
    first_day["ending_stock"] * first_day["holding_cost"]
)

first_day["stockout_penalty"] = (
    first_day["missed_sales"] * first_day["stockout_cost"]
)

first_day["reward"] = (
    first_day["revenue"]
    - first_day["holding_cost_total"]
    - first_day["stockout_penalty"]
)

columns = [
    "product", "demand", "starting_stock", "units_sold",
    "missed_sales", "ending_stock", "revenue",
    "holding_cost_total", "stockout_penalty", "reward"
]

In [ ]:
display(first_day[columns].sort_values("product"))

In [ ]:
print("Total revenue:", first_day["revenue"].sum())
print("Total holding cost:", first_day["holding_cost_total"].sum())
print("Total stockout penalty:", first_day["stockout_penalty"].sum())
print("Total Day 1 reward:", first_day["reward"].sum())

**Experimental assumptions — not real shop records**

In [ ]:
INITIAL_CASH = 10_000
# Calculate the purchase value of the starting inventory
stock_value = products.merge(
    demand[["product", "purchase_cost"]].drop_duplicates(),
    on="product",
    how="left"
)

stock_value["initial_stock_value"] = (
    stock_value["initial_inventory"] * stock_value["purchase_cost"]
)

display(stock_value[
    ["product", "initial_inventory", "purchase_cost",
     "initial_stock_value"]
])

In [ ]:
print("Provisional starting cash:", INITIAL_CASH)
print("Initial stock purchase value:",
      stock_value["initial_stock_value"].sum())

In [ ]:
order_check = suppliers.merge(
    demand[["product", "purchase_cost"]].drop_duplicates(),
    on="product",
    how="left"
)

order_check["minimum_order_cost"] = (
    order_check["minimum_order_quantity"]
    * order_check["purchase_cost"]
    + order_check["fixed_order_cost"]
)

display(order_check[[
    "product",
    "minimum_order_quantity",
    "purchase_cost",
    "fixed_order_cost",
    "minimum_order_cost"
]])

In [ ]:
print("Starting cash:", INITIAL_CASH)

**first simulator**

In [ ]:
PRODUCTS = ["Cooking Oil", "Rice", "Soap"]
INITIAL_CASH = 10_000

# Build lookup tables from your CSV data
product_info = products.set_index("product").to_dict("index")
supplier_info = suppliers.set_index("product").to_dict("index")

daily_data = demand.copy()
daily_data["date"] = pd.to_datetime(daily_data["date"])
#Each date becomes one row, with one set of values per product
days = sorted(daily_data["date"].unique())

**INVENTORY ENVIRONMENT**

In [ ]:
class InventoryEnv:
    def __init__(self, episode_days=None):
        # Use all 90 days unless a date range is provided.
        self.episode_days = list(days if episode_days is None else episode_days)
        self.reset()


    def reset(self):
        self.day_idx = 0
        self.cash = float(INITIAL_CASH)
        self.inventory = {
            p: int(product_info[p]["initial_inventory"])
            for p in PRODUCTS
        }
        self.pending_orders = []
        self.total_reward = 0.0
        return self.get_state()

    def get_state(self):
        return {
            "day": self.day_idx + 1,
            "cash": round(self.cash, 2),
            "inventory": self.inventory.copy(),
            "incoming": {
                p: sum(
                    o["quantity"] for o in self.pending_orders
                    if o["product"] == p
                )
                for p in PRODUCTS
            }
        }

    def step(self, requested_orders):
        if self.day_idx >= len(self.episode_days):
            raise RuntimeError("Episode finished. Call reset().")

        day = self.episode_days[self.day_idx]
        today = daily_data[daily_data["date"] == day]
        today = today.set_index("product")

        revenue = 0.0
        purchase_cost_total = 0.0
        order_fees = 0.0
        holding_cost_total = 0.0
        stockout_penalty = 0.0
        units_sold = {}
        missed_sales = {}

        # 1. Receive orders due today
        arriving = [
            o for o in self.pending_orders
            if o["arrival_day"] <= self.day_idx
        ]
        self.pending_orders = [
            o for o in self.pending_orders
            if o["arrival_day"] > self.day_idx
        ]

        for order in arriving:
            p = order["product"]
            self.inventory[p] += order["quantity"]
       
        # 2. Meet customer demand and calculate today's revenue/costs
        for p in PRODUCTS:
            row = today.loc[p]
            available = self.inventory[p]
            sold = min(available, int(row["demand"]))
            missed = int(row["demand"]) - sold
            order_results = []
            
            self.inventory[p] -= sold
            units_sold[p] = sold
            missed_sales[p] = missed
            revenue += sold * float(row["selling_price"])
            holding_cost_total += (
                self.inventory[p] * float(row["holding_cost"])
            )
            stockout_penalty += (
                missed * float(row["stockout_cost"])
            )
        self.cash += revenue  
        # 3. Place orders at the end of the day.
        for p in PRODUCTS:
            qty = int(requested_orders.get(p, 0))

            # Start by assuming the order was not accepted.
            status = "rejected"
            reason = ""
            order_cost = 0.0

            if qty <= 0:
                status = "no_order"
                reason = "No positive quantity requested"

            else:
                info = product_info[p]
                supplier = supplier_info[p]
                unit_cost = float(today.loc[p, "purchase_cost"])
                fee = float(supplier["fixed_order_cost"])

                if qty < int(supplier["minimum_order_quantity"]):
                    reason = "Below minimum order quantity"

                elif qty > int(info["maximum_order_quantity"]):
                    reason = "Above maximum order quantity"

                else:
                    pipeline_qty = sum(
                        o["quantity"] for o in self.pending_orders
                        if o["product"] == p
                    )

                    if (
                        self.inventory[p] + pipeline_qty + qty
                        > int(info["maximum_stock"])
                    ):
                        reason = "Exceeds maximum storage capacity"

                    else:
                        order_cost = qty * unit_cost + fee

                        if order_cost > self.cash:
                            reason = "Insufficient cash"

                        else:
                            # The order passes every check.
                            self.cash -= order_cost
                            purchase_cost_total += qty * unit_cost
                            order_fees += fee

                            lead_time = int(supplier["lead_time_days"])

                            self.pending_orders.append({
                                "product": p,
                                "quantity": qty,
                                "arrival_day": self.day_idx + lead_time
                            })

                            status = "accepted"
                            reason = "All checks passed"

            # Save the outcome, even if no order was placed.
            order_results.append({
                "product": p,
                "requested_quantity": qty,
                "status": status,
                "reason": reason,
                "order_cost": order_cost
            })

        
        # 4. Calculate reward
        reward = (
            revenue
            - purchase_cost_total
            - order_fees
            - holding_cost_total
            - stockout_penalty
        )

        self.total_reward += reward
        self.day_idx += 1
        done = self.day_idx >= len(self.episode_days)

        info = {
            "date": str(pd.Timestamp(day).date()),
            "revenue": revenue,
            "purchase_cost": purchase_cost_total,
            "order_fees": order_fees,
            "holding_cost": holding_cost_total,
            "stockout_penalty": stockout_penalty,
            "units_sold": units_sold,
            "missed_sales": missed_sales,
            "reward": reward,
            "order_results": order_results,
        }

        return self.get_state(), reward, done, info


print("Inventory environment defined successfully.")

In [ ]:
# Get all unique dates from the demand dataset
days = sorted(demand["date"].unique())

# First 60 days are for training
train_days = days[:60]

# Last 30 days are for testing
test_days = days[60:]

In [ ]:
print("Total days:", len(days))
print("Training days:", len(train_days))
print("Testing days:", len(test_days))

print("Training starts:", train_days[0])
print("Training ends:", train_days[-1])

print("Testing starts:", test_days[0])
print("Testing ends:", test_days[-1])

In [ ]:
env = InventoryEnv(episode_days=train_days)

state = env.reset()

for day_number in range(60):
    orders = {p: 0 for p in PRODUCTS}

    next_state, reward, done, info = env.step(orders)

    if day_number in [0, 58, 59]:
        print(
            "Day:", day_number + 1,
            "| Done:", done
        )

print("Final day index:", env.day_idx)
print("Episode finished:", done)

In [ ]:

env = InventoryEnv()

state = env.get_state()

**Request 10 units of Rice on Day 1.**

In [ ]:
next_state, reward, done, info = env.step({
    "Cooking Oil": 0,
    "Rice": 10,
    "Soap": 0
})

In [ ]:

print("Order tracking results:")
for result in info["order_results"]:
    print(result)

**Test the environment with no orders**

In [ ]:
env = InventoryEnv()
state = env.reset()
next_state, reward, done, info = env.step({
    "Cooking Oil": 0,
    "Rice": 0,
    "Soap": 0
})

In [ ]:
print("Day details:")
display(pd.DataFrame([info]))

print("Next state:")
print(next_state)

print("\nReward:", reward)
print("Cash remaining:", next_state["cash"])

**# Fresh test environment**

In [ ]:
test_env = InventoryEnv()
state = test_env.reset()

**Day 1: order 10 units of Rice**

In [ ]:
state, reward, done, info = test_env.step({
    "Cooking Oil": 0,
    "Rice": 10,
    "Soap": 0
})

In [ ]:
print("After Day 1:")
print("Cash:", state["cash"])
print("Inventory:", state["inventory"])
print("Incoming orders:", state["incoming"])
print("Day 1 purchase cost:", info["purchase_cost"])
print("Day 1 order fee:", info["order_fees"])

**Day 2: place no additional orders**

In [ ]:
state, reward, done, info = test_env.step({
    "Cooking Oil": 0,
    "Rice": 0,
    "Soap": 0
})

In [ ]:
print("\nAfter Day 2:")
print("Inventory:", state["inventory"])
print("Incoming orders:", state["incoming"])
print("Day 2 sales:", info["units_sold"])

**Test 1: Rice order exceeds maximum order quantity (150)**

In [ ]:
env = InventoryEnv()
state, reward, done, info = env.step({
    "Cooking Oil": 0, "Rice": 151, "Soap": 0
})

In [ ]:
print("Test 1 — Excessive Rice order")
print("Rice purchase cost:", info["purchase_cost"])
print("Rice incoming:", state["incoming"]["Rice"])

**Test 2: Cooking Oil order exceeds available storage capacity**

In [ ]:
env = InventoryEnv()
state, reward, done, info = env.step({
    "Cooking Oil": 100, "Rice": 0, "Soap": 0
})

In [ ]:
print("\nTest 2 — Cooking Oil storage limit")
print("Cooking Oil purchase cost:", info["purchase_cost"])
print("Cooking Oil incoming:", state["incoming"]["Cooking Oil"])

**# Test 3: Rice order exceeds available cash**

In [ ]:
env = InventoryEnv()
state, reward, done, info = env.step({
    "Cooking Oil": 0, "Rice": 150, "Soap": 0
})

In [ ]:
print("\nTest 3 — Unaffordable Rice order")
print("Rice purchase cost:", info["purchase_cost"])
print("Rice incoming:", state["incoming"]["Rice"])
print("Cash:", state["cash"])

**Test 4: Properly test the cash constraint**

In [ ]:
env = InventoryEnv()
# Artificial test setup: low cash, not the normal project scenario
env.cash = 1000

state, reward, done, info = env.step({
    "Cooking Oil": 0,
    "Rice": 80,
    "Soap": 0
})

In [ ]:
print("Test 4 — Insufficient cash")
print("Purchase cost:", info["purchase_cost"])
print("Order fees:", info["order_fees"])
print("Cash after the day:", state["cash"])
print("Rice incoming:", state["incoming"]["Rice"])

**Build the rule-based inventory policy**

**Calculate average demand and demand variability**

In [ ]:
demand_summary = (
    demand.groupby("product")["demand"]
    .agg(
        average_daily_demand="mean",
        demand_std="std",
        minimum_daily_demand="min",
        maximum_daily_demand="max"
    )
    .reset_index()
)

**Add supplier delivery times**

In [ ]:
supplier_summary = suppliers[
    ["product", "lead_time_days"]
].copy()

# Combine demand statistics with delivery times
policy_table = demand_summary.merge(
    supplier_summary,
    on="product",
    how="left"
)

**Provisional safety stock: one standard deviation**

In [ ]:
policy_table["safety_stock"] = np.ceil(
    policy_table["demand_std"]
).astype(int)

**Reorder point = average demand during lead time + safety stock**

In [ ]:
policy_table["reorder_point"] = np.ceil(
    policy_table["average_daily_demand"]
    * policy_table["lead_time_days"]
    + policy_table["safety_stock"]
).astype(int)

In [ ]:
print(policy_table.round(2).to_string(index=False))

**Rule-based inventory policy: Version 1**

*Convert the table into a convenient lookup*

In [ ]:
policy_info = policy_table.set_index("product").to_dict("index")

def rule_based_policy(state):
    requested_orders = {}

    for product in PRODUCTS:
        # Current stock on the shelf
        on_hand = state["inventory"][product]

        # Stock already ordered but not yet delivered
        incoming = state["incoming"][product]

        # Inventory position = stock on shelf + incoming stock
        inventory_position = on_hand + incoming

        # Information about this product
        rule = policy_info[product]
        product_limits = product_info[product]

        reorder_point = int(rule["reorder_point"])
        average_demand = float(rule["average_daily_demand"])
        lead_time = int(rule["lead_time_days"])

        # Target stock: reorder point + expected demand during lead time
        target_stock = int(np.ceil(
            reorder_point + average_demand * lead_time
        ))

        # Order only when inventory position is at/below reorder point
        if inventory_position <= reorder_point:
            quantity_needed = target_stock - inventory_position

            # Respect maximum order quantity
            quantity_needed = min(
                quantity_needed,
                int(product_limits["maximum_order_quantity"])
            )

            # Respect maximum storage capacity, including incoming stock
            storage_room = (
                int(product_limits["maximum_stock"])
                - inventory_position
            )

            quantity_needed = min(quantity_needed, storage_room)

            requested_orders[product] = max(0, quantity_needed)
        else:
            requested_orders[product] = 0

    return requested_orders

**Test the policy on the initial state**

In [ ]:
env = InventoryEnv()
initial_state = env.get_state()

In [ ]:
print("Initial inventory:", initial_state["inventory"])
print("Initial incoming orders:", initial_state["incoming"])
print("Rule-based order suggestions:", rule_based_policy(initial_state))

**Run the rule-based policy for all 90 days**

In [ ]:
env = InventoryEnv()
baseline_history = []

while True:
    # Observe the current shop situation
    state = env.get_state()

    # Decide what to order
    orders = rule_based_policy(state)

    # Simulate one day
    next_state, reward, done, info = env.step(orders)

    # Save that day's results
    baseline_history.append({
        "date": info["date"],
        "orders_requested": orders,
        "revenue": info["revenue"],
        "purchase_cost": info["purchase_cost"],
        "order_fees": info["order_fees"],
        "holding_cost": info["holding_cost"],
        "stockout_penalty": info["stockout_penalty"],
        "reward": reward,
        "cash": next_state["cash"],
        "missed_sales_total": sum(info["missed_sales"].values()),
        "units_sold_total": sum(info["units_sold"].values())
    })

    if done:
        break

* Re-run the rule-based policy for all 90 days
*

In [ ]:
env = InventoryEnv()

baseline_history = []
order_log = []

In [ ]:
while True:
    state = env.get_state()
    orders = rule_based_policy(state)

    next_state, reward, done, info = env.step(orders)

    # Save the daily business results.
    baseline_history.append({
        "date": info["date"],
        "revenue": info["revenue"],
        "purchase_cost": info["purchase_cost"],
        "order_fees": info["order_fees"],
        "holding_cost": info["holding_cost"],
        "stockout_penalty": info["stockout_penalty"],
        "reward": reward,
        "cash": next_state["cash"],
        "missed_sales_total": sum(info["missed_sales"].values()),
        "units_sold_total": sum(info["units_sold"].values())
    })

    # Save every product's order outcome.
    for result in info["order_results"]:
        order_log.append({
            "date": info["date"],
            **result
        })

    if done:
        break


In [ ]:
baseline_df = pd.DataFrame(baseline_history)
orders_df = pd.DataFrame(order_log)

print("Days simulated:", len(baseline_df))
print("Order records:", len(orders_df))

print("\nOrder status counts:")
print(orders_df["status"].value_counts())

print("\nOrder results by product:")
print(
    orders_df.groupby(["product", "status"])
    .size()
    .unstack(fill_value=0)
)

print("\nRejected order reasons:")
rejected = orders_df[orders_df["status"] == "rejected"]

if len(rejected) > 0:
    print(rejected["reason"].value_counts())
else:
    print("No orders were rejected.")

**Convert daily records into a table**

In [ ]:
baseline_df = pd.DataFrame(baseline_history)

In [ ]:
print("Days simulated:", len(baseline_df))
print("Total reward:", round(baseline_df["reward"].sum(), 2))
print("Total revenue:", round(baseline_df["revenue"].sum(), 2))
print("Total purchase costs:", round(baseline_df["purchase_cost"].sum(), 2))
print("Total order fees:", round(baseline_df["order_fees"].sum(), 2))
print("Total holding costs:", round(baseline_df["holding_cost"].sum(), 2))
print("Total stockout penalties:", round(baseline_df["stockout_penalty"].sum(), 2))
print("Total missed sales (units):", baseline_df["missed_sales_total"].sum())
print("Ending cash:", round(baseline_df.iloc[-1]["cash"], 2))

In [ ]:
baseline_df.head(10)

In [ ]:
baseline_df.tail(10)

**Analyze baseline performance by product**

In [ ]:
product_results = []
for product in PRODUCTS:
    product_demand = demand[demand["product"] == product]["demand"].sum()

    # Sum the daily missed-sales dictionaries
    missed = sum(
        row["missed_sales_total"]
        for row in baseline_history
    )

    # Get total units sold and missed for this specific product
    product_missed = 0
    product_sold = 0

    # Re-run the baseline to collect product-specific results
    # (This creates a fresh simulation and does not change the saved baseline.)
    break


*Re-run the baseline and record results separately for each product*****

In [ ]:
env = InventoryEnv()
product_history = []

while True:
    state = env.get_state()
    orders = rule_based_policy(state)
    next_state, reward, done, info = env.step(orders)

    record = {
        "date": info["date"],
        "reward": reward
    }

    for product in PRODUCTS:
        record[f"{product}_sold"] = info["units_sold"][product]
        record[f"{product}_missed"] = info["missed_sales"][product]

    product_history.append(record)

    if done:
        break

In [ ]:
# Create a new environment using only the 30 test days
baseline_test_env = InventoryEnv(episode_days=test_days)

baseline_test_product_history = []

while True:
    state = baseline_test_env.get_state()
    orders = rule_based_policy(state)

    next_state, reward, done, info = baseline_test_env.step(orders)

    record = {
        "date": info["date"],
        "reward": reward
    }

    for product in PRODUCTS:
        record[f"{product}_sold"] = info["units_sold"][product]
        record[f"{product}_missed"] = info["missed_sales"][product]

    baseline_test_product_history.append(record)

    if done:
        break

baseline_test_product_df = pd.DataFrame(baseline_test_product_history)

print("Number of test days:", len(baseline_test_product_df))
print("First date:", baseline_test_product_df["date"].iloc[0])
print("Last date:", baseline_test_product_df["date"].iloc[-1])

display(baseline_test_product_df.head())

In [ ]:
# Calculate total sales and missed demand for each product
baseline_product_summary = []

for product in PRODUCTS:
    total_demand = demand[
        (demand["product"] == product) &
        (demand["date"].isin(pd.to_datetime(test_days)))
    ]["demand"].sum()

    units_sold = baseline_test_product_df[f"{product}_sold"].sum()
    missed_units = baseline_test_product_df[f"{product}_missed"].sum()

    fill_rate = (units_sold / total_demand) * 100

    baseline_product_summary.append({
        "Product": product,
        "Total Demand": total_demand,
        "Units Sold": units_sold,
        "Missed Units": missed_units,
        "Fill Rate (%)": round(fill_rate, 2)
    })

baseline_product_summary_df = pd.DataFrame(baseline_product_summary)

display(baseline_product_summary_df)

print("\nTOTALS")
print("Demand:", baseline_product_summary_df["Total Demand"].sum())
print("Units sold:", baseline_product_summary_df["Units Sold"].sum())
print("Missed units:", baseline_product_summary_df["Missed Units"].sum())

In [ ]:
product_df = pd.DataFrame(product_history)

for product in PRODUCTS:
    sold = product_df[f"{product}_sold"].sum()
    missed = product_df[f"{product}_missed"].sum()
    print(f"\n{product}")
    print("Units sold:", sold)
    print("Missed units:", missed)

In [ ]:
print("\nTotal missed units:",
      sum(product_df[f"{p}_missed"].sum() for p in PRODUCTS))

**measure the missed-sales rate**

*Calculate how much demand we missed for each product*

In [ ]:
for product in PRODUCTS:
    total_demand = (
        product_df[f"{product}_sold"].sum()
        + product_df[f"{product}_missed"].sum()
    )

    total_sold = product_df[f"{product}_sold"].sum()
    total_missed = product_df[f"{product}_missed"].sum()

    missed_rate = (total_missed / total_demand) * 100
    fill_rate = (total_sold / total_demand) * 100

    print(f"\n{product}")
    print(f"Total demand: {total_demand}")
    print(f"Units sold: {total_sold}")
    print(f"Missed units: {total_missed}")
    print(f"Missed-demand rate: {missed_rate:.2f}%")
    print(f"Demand fill rate: {fill_rate:.2f}%")


**# Measure missed units and stockout penalties by product**

In [ ]:
product_results = []

for product in PRODUCTS:
    total_sold = product_df[f"{product}_sold"].sum()
    total_missed = product_df[f"{product}_missed"].sum()

    product_demand = total_sold + total_missed

    # Get the product's stockout penalty per missed unit.
    stockout_cost = float(
        demand.loc[
            demand["product"] == product,
            "stockout_cost"
        ].iloc[0]
    )

    product_results.append({
        "product": product,
        "total_demand": int(product_demand),
        "units_sold": int(total_sold),
        "missed_units": int(total_missed),
        "missed_demand_rate_pct": round(
            total_missed / product_demand * 100, 2
        ),
        "stockout_penalty_per_unit": stockout_cost,
        "total_stockout_penalty": round(
            total_missed * stockout_cost, 2
        )
    })

In [ ]:
product_results_df = pd.DataFrame(product_results)

print(product_results_df.to_string(index=False))


**What should our Q-learning agent try to improve?
Imagine training a shop assistant who makes inventory decisions every day. The assistant should learn to:
- Order early enough to reduce missed sales.
- Avoid ordering too much and filling the storage space.
- Keep enough cash to afford purchases.
- Consider supplier delivery delays.
- Balance stockout penalties against purchase, ordering, and holding costs.**

In [ ]:

print("Number of unique dates:", len(days))
print("First date:", pd.Timestamp(days[0]).date())
print("Last date:", pd.Timestamp(days[-1]).date())

print("\nFirst 60-day training period ends:",
      pd.Timestamp(days[59]).date())

print("Last 30-day test period starts:",
      pd.Timestamp(days[60]).date())


**Divide the 90 dates into training and testing periods.**

In [ ]:
train_days = days[:60]
test_days = days[60:]

# Create one simulator for each period.
train_env = InventoryEnv(episode_days=train_days)
test_env = InventoryEnv(episode_days=test_days)

In [ ]:
print("Training days:", len(train_env.episode_days))
print("Training starts:", pd.Timestamp(train_days[0]).date())
print("Training ends:", pd.Timestamp(train_days[-1]).date())

print("\nTesting days:", len(test_env.episode_days))
print("Testing starts:", pd.Timestamp(test_days[0]).date())
print("Testing ends:", pd.Timestamp(test_days[-1]).date())

# Check that each simulator begins with the same initial inventory.
print("\nTraining initial inventory:", train_env.inventory)
print("Testing initial inventory:", test_env.inventory)


In [ ]:

def get_stock_level(product, on_hand, incoming):
    # Count stock on the shelf and stock already on the way.
    inventory_position = on_hand + incoming

    rule = policy_info[product]
    reorder_point = int(rule["reorder_point"])

    average_demand = float(rule["average_daily_demand"])
    lead_time = int(rule["lead_time_days"])

    target_stock = int(
        np.ceil(reorder_point + average_demand * lead_time)
    )

    if inventory_position <= reorder_point:
        return "low"
    elif inventory_position < target_stock:
        return "medium"
    else:
        return "high"


In [ ]:
for product in PRODUCTS:
    on_hand = train_env.inventory[product]
    incoming = 0

    level = get_stock_level(product, on_hand, incoming)

    print(
        f"{product}: on-hand={on_hand}, "
        f"incoming={incoming}, stock level={level}"
    )


**define the agent's actions
 Let's give it four choices for each product:
- 0 — order nothing.
- 1 — order a small quantity.
- 2 — order a medium quantity.
- 3 — order a large quantity.**

*# Define four possible order actions for each product.*

In [ ]:
ACTION_LEVELS = {
    0: "No order",
    1: "Small order",
    2: "Medium order",
    3: "Large order"
}

In [ ]:
def get_order_quantity(product, action):
    average_demand = float(
        policy_info[product]["average_daily_demand"]
    )
    lead_time = int(
        policy_info[product]["lead_time_days"]
    )

    if action == 0:
        return 0
    elif action == 1:
        return max(1, int(np.ceil(average_demand)))
    elif action == 2:
        return max(1, int(np.ceil(average_demand * lead_time * 2)))
    elif action == 3:
        return max(1, int(np.ceil(average_demand * lead_time * 3)))
    else:
        raise ValueError("Action must be 0, 1, 2, or 3.")


*# Show the quantity represented by each action.*

In [ ]:
for product in PRODUCTS:
    print(f"\n{product}")
    for action, label in ACTION_LEVELS.items():
        quantity = get_order_quantity(product, action)
        print(f"  {action} - {label}: {quantity} units")

**Check which action quantities are valid**

*# Check each action quantity against the product limits.*

In [ ]:
for product in PRODUCTS:
    info = product_info[product]
    supplier = supplier_info[product]

    print(f"\n{product}")
    print("Minimum supplier order:",
          int(supplier["minimum_order_quantity"]))
    print("Maximum order quantity:",
          int(info["maximum_order_quantity"]))
    print("Maximum stock:",
          int(info["maximum_stock"]))

    for action, label in ACTION_LEVELS.items():
        qty = get_order_quantity(product, action)

        order_size_ok = (
            qty == 0 or
            int(supplier["minimum_order_quantity"])
            <= qty
            <= int(info["maximum_order_quantity"])
        )

        print(
            f"  {label}: {qty} units | "
            f"Order-size valid: {order_size_ok}"
        )


In [ ]:

from itertools import product as cartesian_product

# Each product can choose action 0, 1, 2, or 3.
ACTION_CHOICES = [0, 1, 2, 3]

# Generate every combination for the three products.
all_actions = list(
    cartesian_product(ACTION_CHOICES, repeat=len(PRODUCTS))
)

In [ ]:
print("Number of combined actions:", len(all_actions))


In [ ]:
print("\nFirst 10 actions:")
for action in all_actions[:10]:
    print({
        product: get_order_quantity(product, choice)
        for product, choice in zip(PRODUCTS, action)
    })

In [ ]:

def get_cash_level(cash):
    if cash < 3000:
        return "low"
    elif cash < 7000:
        return "medium"
    else:
        return "high"

In [ ]:
def get_time_level(day_number, total_days):
    progress = day_number / total_days

    if progress <= 1 / 3:
        return "early"
    elif progress <= 2 / 3:
        return "middle"
    else:
        return "late"

In [ ]:
def get_q_state(state, total_days):
    # Categorize stock for each product.
    stock_levels = tuple(
        get_stock_level(
            product,
            state["inventory"][product],
            state["incoming"][product]
        )
        for product in PRODUCTS
    )

    # Categorize cash and time.
    cash_level = get_cash_level(state["cash"])
    time_level = get_time_level(state["day"], total_days)

    return stock_levels + (cash_level, time_level)

In [ ]:
# Test the state representation using the initial training state.
initial_state = train_env.get_state()
q_state = get_q_state(initial_state, len(train_env.episode_days))

print("Initial shop state:")
print(q_state)
print("Number of state components:", len(q_state))

**Create the Q-table**

*# Create an empty Q-table for the learning agent.
*

In [ ]:
from collections import defaultdict

q_table = defaultdict(
    lambda: np.zeros(len(all_actions), dtype=float)
)

*# The first time we access a state, all 64 action scores are zero*

In [ ]:
example_state = get_q_state(
    train_env.get_state(),
    len(train_env.episode_days)
)

In [ ]:
print("Example state:", example_state)
print("Number of available actions:", len(all_actions))
print("Initial Q-values:", q_table[example_state])
print("Number of states stored so far:", len(q_table))

In [ ]:
import random

def choose_action(state, epsilon):
    # Explore: randomly try an action.
    if random.random() < epsilon:
        return random.randrange(len(all_actions))

    # Exploit: choose an action with the highest estimated Q-value.
    q_values = q_table[state]
    best_value = np.max(q_values)

    # If several actions tie, choose randomly among the best.
    best_actions = np.flatnonzero(q_values == best_value)

    return int(random.choice(best_actions))


**# Test exploration and exploitation.**

In [ ]:
test_state = get_q_state(
    train_env.get_state(),
    len(train_env.episode_days)
)

In [ ]:
print("Random exploratory action:", choose_action(test_state, epsilon=1.0))
print("Greedy action:", choose_action(test_state, epsilon=0.0))
print("Number of actions:", len(all_actions))

**# A small practice example of the Q-learning update.
**

In [ ]:
learning_rate = 0.1
discount_factor = 0.95

In [ ]:

old_q = 0.0
reward = 100.0
best_future_q = 20.0

new_q = old_q + learning_rate * (
    reward + discount_factor * best_future_q - old_q
)

In [ ]:
print("Old Q-value:", old_q)
print("Immediate reward:", reward)
print("Best future Q-value:", best_future_q)
print("Updated Q-value:", new_q)

**Train the Q-learning agent
- State: How much stock and cash does the shop have?
- Action: Should it order nothing, a small amount, a medium amount, or a large amount?
- Reward: Did the decision help the shop earn money and avoid stockouts?
- Learning: After every day, update the Q-table based on the result.**

**# STEP 1: Train our Q-learning inventory agent**

In [ ]:
print("Training days:", len(train_days))

test_env = InventoryEnv(episode_days=train_days)

state = test_env.reset()

print("Starting day index:", test_env.day_idx)
print("Episode days:", len(test_env.episode_days))

In [ ]:
import inspect

print(inspect.getsource(InventoryEnv.step))

In [ ]:
from collections import defaultdict
import numpy as np

In [ ]:
# Learning settings
learning_rate = 0.1
discount_factor = 0.95
episodes = 500

# The Q-table starts with zero values for unseen states
q_table = defaultdict(
    lambda: np.zeros(len(all_actions), dtype=float)
)

In [ ]:
# Start a fresh Q-table
q_table = defaultdict(
    lambda: np.zeros(len(all_actions), dtype=float)
)

*# Create one training environment*

In [ ]:
env = InventoryEnv(episode_days=train_days)
state_dict = env.reset()

# Get the first state
state = get_q_state(state_dict, len(train_days))

print("Initial state:", state)
print("Q-table before training:", len(q_table))

In [ ]:
# Choose an action
action_index = choose_action(state, 1.0)
action_tuple = all_actions[action_index]

requested_orders = {
    product: get_order_quantity(product, action)
    for product, action in zip(PRODUCTS, action_tuple)
}

In [ ]:
# Execute one simulation step
next_state_dict, reward, done, info = env.step(requested_orders)

# Update the Q-table with this state
old_q = q_table[state][action_index]

best_future_q = 0.0

new_q = old_q + learning_rate * (
    reward + discount_factor * best_future_q - old_q
)

q_table[state][action_index] = new_q


In [ ]:
print("\nReward:", reward)
print("Q-table after one update:", len(q_table))
print("Updated Q-value:", q_table[state][action_index])

In [ ]:
print("Learning rate:", learning_rate)
print("Discount factor:", discount_factor)
print("Reward:", reward)
print("Updated Q-value:", q_table[state][action_index])
print("Selected action:", action_tuple)
print("Q-table size:", len(q_table))

In [ ]:
print("Old Q-value before update:", old_q)
print("Learning rate:", learning_rate)
print("Reward:", reward)
print("Discount factor:", discount_factor)
print("Best future Q-value:", best_future_q)

expected_q = old_q + learning_rate * (
    reward + discount_factor * best_future_q - old_q
)

print("Expected Q-value:", expected_q)
print("Actual Q-value:", q_table[state][action_index])

In [ ]:
# Create a completely fresh Q-table
q_table = defaultdict(
    lambda: np.zeros(len(all_actions), dtype=float)
)

# Reset training history
training_rewards = []

print("Fresh Q-table size:", len(q_table))
print("Starting training...")

# Run the complete training loop here

In [ ]:
training_rewards = []

# Repeat training 500 times using the first 60 days
for episode in range(episodes):

    # Start each training episode with fresh initial conditions
    env = InventoryEnv(episode_days=train_days)
    env.reset()
    state_dict = env.get_state()

    # Explore more at the beginning, then gradually explore less
    epsilon = max(0.05, 1.0 * (0.99 ** episode))

    total_reward = 0

    while True:

        # Convert shop information into our simplified Q-learning state
        state = get_q_state(state_dict, len(train_days))

        # Choose one of the 64 possible combinations of order actions
        action_index = choose_action(state, epsilon)
        action_tuple = all_actions[action_index]

        # Convert action codes into order quantities for each product
        requested_orders = {
            product: get_order_quantity(product, action)
            for product, action in zip(PRODUCTS, action_tuple)
        }

        # Let the simulator process deliveries, sales, and orders
        next_state_dict, reward, done, info = env.step(requested_orders)

        # Convert the resulting shop information into the next state
        next_state = get_q_state(
            next_state_dict, len(train_days)
        )

        # Read the old Q-value for the action we selected
        old_q = q_table[state][action_index]
        if episode == 0 and len(q_table) <= 3:
            print("DEBUG:")
            print("Episode:", episode)
            print("State:", state)
            print("Q-table size:", len(q_table))
        # At the end of an episode, there is no future reward
        if done:
            best_future_q = 0.0
        else:
            best_future_q = np.max(q_table[next_state])

        # The Q-learning update
        new_q = old_q + learning_rate * (
            reward
            + discount_factor * best_future_q
            - old_q
        )

        q_table[state][action_index] = new_q
        if episode == 0 and len(q_table) <= 3:
            print("Q-table size after update:", len(q_table))
        total_reward += reward
        state_dict = next_state_dict

        if done:
            break

    training_rewards.append(total_reward)

    # Show progress every 100 episodes
    if (episode + 1) % 100 == 0:
        recent_average = np.mean(training_rewards[-100:])
        print(
            f"Episode {episode + 1}/{episodes} | "
            f"Average reward over last 100 episodes: "
            f"{recent_average:.2f} | "
            f"Epsilon: {epsilon:.3f}"
        )

In [ ]:
print("\nCHECKPOINT AFTER TRAINING")
print("Q-table size:", len(q_table))
print("Q-table object ID:", id(q_table))
print("Training episodes:", len(training_rewards))

In [ ]:
print("\nFINAL Q-TABLE CHECK")
print("Q-table size:", len(q_table))
print("Q-table object ID:", id(q_table))
print("Training episodes:", len(training_rewards))

In [ ]:
print("Q-table size before evaluation:", len(q_table))
print("Q-table object ID:", id(q_table))

In [ ]:
print("Q-table size:", len(q_table))
print("Q-table object ID:", id(q_table))
# Check the trained Q-table before evaluation

print("Learned Q-table states:", len(q_table))

env = InventoryEnv(episode_days=test_days)
state_dict = env.reset()

seen_states = 0
unseen_states = 0

for day in range(len(test_days)):

    state = get_q_state(state_dict, len(train_days))

    if state in q_table:
        seen_states += 1
    else:
        unseen_states += 1

    # Move forward using a no-order action only to inspect
    # the states encountered along this simulated path
    next_state_dict, reward, done, info = env.step(
        {product: 0 for product in PRODUCTS}
    )

    state_dict = next_state_dict

    if done:
        break

print("\nTest-state coverage:")
print("Seen states:", seen_states)
print("Unseen states:", unseen_states)
print("Total:", seen_states + unseen_states)

In [ ]:
print("Learned Q-table states:", len(q_table))

env = InventoryEnv(episode_days=test_days)
state_dict = env.reset()

seen_states = 0
unseen_states = 0

for day in range(len(test_days)):

    state = get_q_state(state_dict, len(train_days))

    if state in q_table:
        seen_states += 1
    else:
        unseen_states += 1

    next_state_dict, reward, done, info = env.step(
        {product: 0 for product in PRODUCTS}
    )

    state_dict = next_state_dict

    if done:
        break

print("\nTest-state coverage:")
print("Seen states:", seen_states)
print("Unseen states:", unseen_states)
print("Total:", seen_states + unseen_states)

In [ ]:
print("Training episodes completed:", len(training_rewards))
print("Learned Q-table states:", len(q_table))
print("First 5 Q-table states:")

for i, state in enumerate(q_table):
    print(state)
    if i == 4:
        break

In [ ]:
print("\nTraining complete!")
print("Episodes completed:", len(training_rewards))
print("States learned:", len(q_table))
print("Final episode reward:", round(training_rewards[-1], 2))


In [ ]:
# Use ONLY the first 60 days to calculate demand statistics

train_demand = demand[
    demand["date"].isin(pd.to_datetime(train_days))
].copy()

demand_summary = (
    train_demand.groupby("product")["demand"]
    .agg(
        average_daily_demand="mean",
        demand_std="std",
        minimum_daily_demand="min",
        maximum_daily_demand="max"
    )
    .reset_index()
)

supplier_summary = suppliers[
    ["product", "lead_time_days"]
].copy()

policy_table = demand_summary.merge(
    supplier_summary,
    on="product",
    how="left"
)

policy_table["safety_stock"] = np.ceil(
    policy_table["demand_std"]
).astype(int)

policy_table["reorder_point"] = np.ceil(
    policy_table["average_daily_demand"]
    * policy_table["lead_time_days"]
    + policy_table["safety_stock"]
).astype(int)

policy_info = policy_table.set_index("product").to_dict("index")

print("Training-only inventory policy:")
display(policy_table)

In [ ]:
q_table = defaultdict(
    lambda: np.zeros(len(all_actions), dtype=float)
)

Test both policies on the same 30 days

**We compare:
1. Rule-based policy: Uses reorder points and safety stock.
2. Q-learning policy: Uses its learned Q-table to choose ordering actions.> > **

*#Both start with the same initial inventory and 10,000 ETB cash, and both face March 2–31 demand. *

# FAIR TEST: Rule-based policy vs Q-learning
# Both use the same 30 test days and initial conditions.***

In [ ]:
def evaluate_policy(policy_name):
    env = InventoryEnv(episode_days=test_days)
    state_dict = env.reset()

    total_reward = 0
    total_revenue = 0
    total_purchase_cost = 0
    total_order_fees = 0
    total_holding_cost = 0
    total_stockout_penalty = 0
    total_units_sold = 0
    total_missed_sales = 0
    total_demand = 0

    unseen_states = 0
    test_states = set()

    while True:

        if policy_name == "Rule-based":

            requested_orders = rule_based_policy(state_dict)
        elif policy_name == "Q-learning":
            state = get_q_state(state_dict, len(train_days))
            test_states.add(state)
            if state not in q_table:
                unseen_states += 1
            else:
                action_index = int(np.argmax(q_table[state]))
                action_tuple = all_actions[action_index]
                requested_orders = {
                    product: get_order_quantity(product, action)
                    for product, action in zip(PRODUCTS, action_tuple)
                }

        else:
            raise ValueError("Unknown policy name")

        next_state, reward, done, info = env.step(requested_orders)

        total_reward += reward
        total_revenue += info["revenue"]
        total_purchase_cost += info["purchase_cost"]
        total_order_fees += info["order_fees"]
        total_holding_cost += info["holding_cost"]
        total_stockout_penalty += info["stockout_penalty"]

        for product in PRODUCTS:
            sold = info["units_sold"][product]
            missed = info["missed_sales"][product]

            total_units_sold += sold
            total_missed_sales += missed
            total_demand += sold + missed

        state_dict = next_state

        if done:
            break

    fill_rate = (
        total_units_sold / total_demand * 100
        if total_demand > 0 else 0
    )

    return {
        "Policy": policy_name,
        "Reward": round(total_reward, 2),
        "Revenue": round(total_revenue, 2),
        "Purchase Cost": round(total_purchase_cost, 2),
        "Order Fees": round(total_order_fees, 2),
        "Holding Cost": round(total_holding_cost, 2),
        "Stockout Penalty": round(total_stockout_penalty, 2),
        "Units Sold": total_units_sold,
        "Missed Sales": total_missed_sales,
        "Fill Rate (%)": round(fill_rate, 2),
        "Ending Cash": round(env.cash, 2),
        "Unseen Q States": unseen_states
    }

*# Evaluate both policies*

In [ ]:
baseline_test_results = evaluate_policy("Rule-based")
qlearning_test_results = evaluate_policy("Q-learning")

# Display comparison
comparison_df = pd.DataFrame([
    baseline_test_results,
    qlearning_test_results
])

display(comparison_df)

In [ ]:
print("Q-table size:", len(q_table))

print("\nQ-value statistics:")

all_q_values = np.concatenate([
    values for values in q_table.values()
])

print("Minimum Q-value:", all_q_values.min())
print("Maximum Q-value:", all_q_values.max())
print("Mean Q-value:", all_q_values.mean())

print("\nNumber of zero Q-values:", np.sum(all_q_values == 0))
print("Total Q-values:", len(all_q_values))

In [ ]:
from collections import Counter

action_counts = Counter()

env = InventoryEnv(episode_days=test_days)
state_dict = env.reset()

while True:
    state = get_q_state(state_dict, len(train_days))

    if state not in q_table:
        action_index = 0
    else:
        action_index = int(np.argmax(q_table[state]))

    action_tuple = all_actions[action_index]

    action_counts.update(action_tuple)

    requested_orders = {
        product: get_order_quantity(product, action)
        for product, action in zip(PRODUCTS, action_tuple)
    }

    next_state, reward, done, info = env.step(requested_orders)
    state_dict = next_state

    if done:
        break

print("Action selection counts:")
for product, counts in zip(PRODUCTS, [None] * len(PRODUCTS)):
    pass

for i, product in enumerate(PRODUCTS):
    print(f"\n{product}:")
    print(action_counts)

In [ ]:
action_counts = {
    product: Counter()
    for product in PRODUCTS
}

env = InventoryEnv(episode_days=test_days)
state_dict = env.reset()

day_count = 0

while True:
    state = get_q_state(state_dict, len(train_days))

    if state not in q_table:
        action_index = 0
    else:
        action_index = int(np.argmax(q_table[state]))

    action_tuple = all_actions[action_index]

    for product, action in zip(PRODUCTS, action_tuple):
        action_counts[product][ACTION_LEVELS[action]] += 1

    requested_orders = {
        product: get_order_quantity(product, action)
        for product, action in zip(PRODUCTS, action_tuple)
    }

    next_state, reward, done, info = env.step(requested_orders)

    state_dict = next_state
    day_count += 1

    if done:
        break

print("Total test days:", day_count)

for product in PRODUCTS:
    print(f"\n{product}:")
    print(action_counts[product])
    print("Total actions:", sum(action_counts[product].values()))

In [ ]:
env = InventoryEnv(episode_days=test_days)
state_dict = env.reset()

first_state = get_q_state(state_dict, len(train_days))

print("First test state:")
print(first_state)

print("\nIs this state learned?")
print(first_state in q_table)

if first_state in q_table:
    q_values = q_table[first_state]

    for i, action in enumerate(all_actions):
        print(
            action,
            "Q-value:",
            round(q_values[i], 3)
        )

In [ ]:
q_values = q_table[first_state]

best_action_index = int(np.argmax(q_values))
best_action = all_actions[best_action_index]

print("Best action:", best_action)
print("Best Q-value:", round(q_values[best_action_index], 3))

print("\nRecommended orders:")
for product, action in zip(PRODUCTS, best_action):
    quantity = get_order_quantity(product, action)
    print(f"{product}: {quantity} units ({ACTION_LEVELS[action]})")

In [ ]:
test_states = [
    ("low", "low", "low", "high", "early"),
    ("medium", "medium", "medium", "high", "early"),
    ("high", "high", "high", "high", "early")
]

for state in test_states:
    print("\nState:", state)

    if state not in q_table:
        print("This state was not learned.")
        continue

    best_index = int(np.argmax(q_table[state]))
    best_action = all_actions[best_index]

    print("Best action:", best_action)

    for product, action in zip(PRODUCTS, best_action):
        quantity = get_order_quantity(product, action)
        print(f"{product}: {quantity} units")

In [ ]:
env = InventoryEnv(episode_days=test_days)
state_dict = env.reset()

print("Cash:", state_dict["cash"])
print("\nInventory position:")

for product in PRODUCTS:
    on_hand = state_dict["inventory"][product]
    incoming = state_dict["incoming"][product]
    position = on_hand + incoming

    stock_level = get_stock_level(
        product, on_hand, incoming
    )

    print(f"\n{product}")
    print("On-hand:", on_hand)
    print("Incoming:", incoming)
    print("Inventory position:", position)
    print("Stock category:", stock_level)

# compare Q-learning with the rule-based policy for this exact state

In [ ]:
env = InventoryEnv(episode_days=test_days)
state_dict = env.reset()

print("RULE-BASED POLICY")
rule_orders = rule_based_policy(state_dict)

for product in PRODUCTS:
    print(f"{product}: {rule_orders[product]} units")

print("\nQ-LEARNING POLICY")
q_state = get_q_state(state_dict, len(train_days))
best_index = int(np.argmax(q_table[q_state]))
best_action = all_actions[best_index]

q_orders = {
    product: get_order_quantity(product, action)
    for product, action in zip(PRODUCTS, best_action)
}

for product in PRODUCTS:
    print(f"{product}: {q_orders[product]} units")

In [ ]:
q_values = q_table[q_state]

print("Current state:", q_state)
print("Selected action:", best_action)

print("\nAlternative actions for each product:")

for product_index, product in enumerate(PRODUCTS):
    print(f"\n{product}")

    for action in range(4):
        alternative = list(best_action)
        alternative[product_index] = action
        alternative = tuple(alternative)

        index = all_actions.index(alternative)

        print(
            ACTION_LEVELS[action],
            "| Order:",
            get_order_quantity(product, action),
            "| Q-value:",
            round(q_values[index], 3)
        )

**check how many states were actually learned**

In [ ]:
print("Theoretical possible states:", 243)
print("States learned:", len(q_table))
print("States not learned:", 243 - len(q_table))

print(
    "Percentage of state space learned:",
    round(len(q_table) / 243 * 100, 2),
    "%"
)

**inspect action coverage**

In [ ]:
print("Learned states:", len(q_table))

total_actions = 0
states_with_all_actions = 0
states_with_partial_actions = 0

for state, values in q_table.items():
    tried_actions = np.count_nonzero(values)

    total_actions += tried_actions

    if tried_actions == 64:
        states_with_all_actions += 1
    elif tried_actions > 0:
        states_with_partial_actions += 1

print("States with all 64 actions nonzero:", states_with_all_actions)
print("States with partially nonzero actions:", states_with_partial_actions)
print("Total nonzero Q-values:", total_actions)
print("Maximum possible Q-values:", len(q_table) * 64)

**training parameters and exploration schedule.**

In [ ]:
print("Learning rate:", learning_rate)
print("Discount factor:", discount_factor)
print("Training episodes:", episodes)

print("\nExploration settings:")
print("Initial epsilon: 1.0")
print("Minimum epsilon: 0.05")
print("Epsilon decay: 0.99")

print("\nFinal epsilon:")
print(max(0.05, 1.0 * (0.99 ** (episodes - 1))))

In [ ]:
print("Available evaluation variables:")

for name in list(globals().keys()):
    if any(word in name.lower() for word in ["result", "comparison", "evaluation"]):
        print(name)

In [ ]:
print("Q-learning test results:")
print(qlearning_test_results)

print("\nComparison table:")
print(comparison_df)

In [ ]:
print("RULE-BASED RESULTS:")
print(baseline_test_results)

print("\nQ-LEARNING RESULTS:")
print(qlearning_test_results)

**Product-level results**

In [ ]:
print("Product results:")
display(product_results_df.head(10))

print("\nProduct results columns:")
print(product_results_df.columns.tolist())

**inspect the product results structure**

In [ ]:
print("Product results shape:")
print(product_results_df.shape)

print("\nProduct results:")
display(product_results_df)

print("\nAvailable evaluation variables:")
for name in list(globals().keys()):
    if "product" in name.lower() or "test" in name.lower():
        print(name)

In [ ]:
print("PRODUCT RESULTS VARIABLES")

print("\nproduct_results:")
print(type(product_results))

print("\nproduct_history:")
print(type(product_history))

print("\nproduct_df:")
print(type(product_df))

print("\nproduct_results_df:")
print(type(product_results_df))

print("\nproduct_history sample:")
print(product_history[:5] if isinstance(product_history, list) else product_history)

In [ ]:
print("PRODUCT HISTORY")

print("Number of records:", len(product_history))
print("First date:", product_history[0]["date"])
print("Last date:", product_history[-1]["date"])

print("\nPRODUCT RESULTS")

print("Number of records:", len(product_results))

if len(product_results) > 0:
    print("First record:", product_results[0])

print("\nPRODUCT DATAFRAME")

display(product_df.head())
print("Shape:", product_df.shape)

print("\nTEST PERIOD")
print("First test date:", min(test_days))
print("Last test date:", max(test_days))
print("Number of test days:", len(test_days))

In [ ]:
# Filter product history to the 30-day test period
test_product_df = product_df[
    pd.to_datetime(product_df["date"]).isin(pd.to_datetime(test_days))
].copy()

print("Test product history:")
print("Number of days:", len(test_product_df))

# Calculate total demand, sales and missed demand
test_product_summary = []

for product in PRODUCTS:
    total_demand = demand[
        (demand["product"] == product) &
        (demand["date"].isin(pd.to_datetime(test_days)))
    ]["demand"].sum()

    units_sold = test_product_df[f"{product}_sold"].sum()
    missed_units = test_product_df[f"{product}_missed"].sum()

    test_product_summary.append({
        "Product": product,
        "Total Demand": total_demand,
        "Units Sold": units_sold,
        "Missed Units": missed_units
    })

test_product_summary_df = pd.DataFrame(test_product_summary)

display(test_product_summary_df)

**identify the source of product_history**

In [ ]:
print("Where was product_history created?")

for name, obj in list(globals().items()):
    if obj is product_history:
        print("Variable:", name)

print("\nCheck product_results generation code:")

the results identical

Because all 30 test states were unseen. Therefore, the hybrid policy used the rule-based policy for every single test day

*Two likely reasons for the poor result:
1. Unseen states: The agent encounters shop situations that were not represented in its 60-day training experience. For those states, our evaluation code deliberately chooses action 0(no order).
2. Untried actions: Even in a state the agent has seen, some actions may still have Q-values of zero. If the actions it tried received negative rewards, np.argmax() may choose an untried action with a zero value. That can also lead to no ordering.*

**Inspect what the Q-table has actually learned**

In [ ]:
from collections import Counter
import numpy as np

In [ ]:
env = InventoryEnv(episode_days=test_days)
state_dict = env.reset()

action_counts = Counter()
seen_states = 0
unseen_states = 0
zero_best_states = 0
negative_best_states = 0

In [ ]:
diagnostic_rows = []

while True:

    # Use the same state conversion as your current evaluation
    state = get_q_state(state_dict, len(train_days))

    if state in q_table:
        seen_states += 1

        q_values = q_table[state]
        action_index = int(np.argmax(q_values))
        best_q = float(np.max(q_values))

        if best_q == 0:
            zero_best_states += 1

        if best_q < 0:
            negative_best_states += 1

        status = "Seen"
    else:
        unseen_states += 1
        action_index = 0
        best_q = None
        status = "Unseen"

    action_tuple = all_actions[action_index]

    action_counts[action_tuple] += 1

    requested_orders = {
        product: get_order_quantity(product, action)
        for product, action in zip(PRODUCTS, action_tuple)
    }

    next_state, reward, done, info = env.step(requested_orders)

    diagnostic_rows.append({
        "Day": info["date"],
        "State status": status,
        "Best Q-value": best_q,
        "Action": action_tuple,
        "Orders": requested_orders,
        "Reward": reward
    })

    state_dict = next_state

    if done:
        break

In [ ]:
print("Total test days:", len(test_days))
print("Seen states:", seen_states)
print("Unseen states:", unseen_states)
print("Seen states with best Q-value = 0:", zero_best_states)
print("Seen states with negative best Q-value:", negative_best_states)

In [ ]:
print("\nAction selection counts:")
for action, count in action_counts.items():
    print(action, ":", count, "days")

print("\nFirst 10 daily decisions:")
display(pd.DataFrame(diagnostic_rows).head(10))

In [ ]:
# Check the trained Q-table before evaluation

print("Learned Q-table states:", len(q_table))

env = InventoryEnv(episode_days=test_days)
state_dict = env.reset()

seen_states = 0
unseen_states = 0

for day in range(len(test_days)):

    state = get_q_state(state_dict, len(train_days))

    if state in q_table:
        seen_states += 1
    else:
        unseen_states += 1

    # Move forward using a no-order action only to inspect
    # the states encountered along this simulated path
    next_state_dict, reward, done, info = env.step(
        {product: 0 for product in PRODUCTS}
    )

    state_dict = next_state_dict

    if done:
        break

print("\nTest-state coverage:")
print("Seen states:", seen_states)
print("Unseen states:", unseen_states)
print("Total:", seen_states + unseen_states)

# investigate the Q-table
Before changing the algorithm, we need to understand why the training states and testing states do not match

In [ ]:
print("Number of states learned:", len(q_table))

print("\nFirst 10 states in the Q-table:")
for i, state in enumerate(q_table.keys()):
    print(state)
    if i == 9:
        break

print("\nInitial test state:")
test_env = InventoryEnv(episode_days=test_days)
test_state_dict = test_env.reset()

first_test_state = get_q_state(
    test_state_dict,
    len(train_days)
)

print(first_test_state)

print("\nIs the first test state in Q-table?")
print(first_test_state in q_table)

**Check whether the training loop actually updated the Q-table**

In [ ]:
print("Q-table type:", type(q_table))
print("Q-table size:", len(q_table))
print("Training episodes:", episodes)
print("Training days:", len(train_days))

print("\nQ-table variable information:")
print("Is q_table empty?", len(q_table) == 0)

In [ ]:
q_values = q_table[state]
old_q = q_values[action_index]

*Check Q-table initialization*

In [ ]:
from collections import defaultdict
import numpy as np

q_table = defaultdict(
    lambda: np.zeros(len(all_actions), dtype=float)
)

print("Q-table initialized:", len(q_table) == 0)